# Một phần tư số dòng không có mã khách: bỏ, điền hay giữ?

Bộ `online-retail-ii`, sheet **`Year 2010-2011`**. Head of CRM chuẩn bị báo cáo quý về khách hàng (số khách, giá trị
đơn trung bình, tỉ lệ quay lại, khách giá trị cao) và nhắn:

> "Mình thấy khoảng 25% dòng không có `Customer ID`. Team mình định **bỏ hết các dòng đó** rồi tính mọi chỉ số trên
> phần còn lại, hoặc **điền** bằng cách nào đó cho đủ. Bạn xem giúp cách nào đúng, và báo cáo quý có bị lệch không?"

Câu trả lời phụ thuộc vào **cơ chế thiếu**: thiếu ngẫu nhiên hoàn toàn (MCAR), thiếu theo các biến quan sát được (MAR),
hay thiếu vì chính giá trị bị thiếu (MNAR). Bạn không chứng minh được cơ chế, nhưng có thể thu thập bằng chứng: dữ liệu
thiếu rải đều hay tập trung theo quốc gia, giờ, kiểu hoá đơn? Hoá đơn thiếu mã khác hoá đơn có mã thế nào?

Phần được chấm tự động: hai hàm hồ sơ (`missing_profile`, `invoice_summary`) và khung memo. Nội dung điều tra và memo
chấm theo rubric trong `SOLUTION.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/advanced/02-missing-customer-id -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011
# xlsx đọc mất 30-60 giây: lần đầu lưu bản parquet vào thư mục tạm, các lần sau đọc trong vài giây
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_sheet(sheet):
    src = ensure("online-retail-ii") / "online_retail_II.xlsx"
    st = src.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"online_retail_II-{sheet.replace(' ', '_')}-{st.st_size}-{st.st_mtime_ns}.parquet"
    if not cache.exists():
        df = pd.read_excel(src, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
        cache.parent.mkdir(parents=True, exist_ok=True)
        tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
        df.to_parquet(tmp, index=False)
        os.replace(tmp, cache)  # ghi atomic: không để lại tệp dở dang
    return pd.read_parquet(cache)


tx = read_sheet("Year 2010-2011")
valid = tx[~tx["Invoice"].astype(str).str.startswith("C") & (tx["Quantity"] > 0) & (tx["Price"] > 0)].copy()
print(tx.shape, "→", valid.shape, "dòng hợp lệ theo định nghĩa dùng chung")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

## 1. Tỉ lệ thiếu theo nhóm

`missing_profile(df, col, by) -> DataFrame`: với mỗi giá trị của cột `by`: `rows` (số dòng), `missing` (số dòng `col`
thiếu), `missing_rate`. Index tên `by`, sắp `rows` giảm dần, bằng nhau thì khoá tăng dần. `rows`, `missing` kiểu số
nguyên.

In [ ]:
def missing_profile(df: pd.DataFrame, col: str, by: str) -> pd.DataFrame:
    """Số dòng, số dòng thiếu `col` và tỉ lệ thiếu theo từng giá trị của `by`."""
    # TODO
    raise NotImplementedError

## 2. Hồ sơ mỗi hoá đơn

Thiếu mã khách là thuộc tính của **dòng** hay của **hoá đơn**? Nếu một hoá đơn có dòng có mã, dòng không mã, thì thiếu
là lỗi nhập từng dòng; nếu cả hoá đơn cùng thiếu, thiếu đến từ cách hoá đơn được tạo ra.

`invoice_summary(lines, col="Customer ID") -> DataFrame`, index `Invoice` (giữ kiểu như đầu vào, sắp tăng dần), cột:

| Cột | Ý nghĩa |
| --- | --- |
| `lines` | số dòng |
| `revenue` | tổng `Quantity * Price` |
| `missing` | mọi dòng của hoá đơn đều thiếu `col` |
| `mixed` | có dòng thiếu và có dòng không thiếu |
| `country` | `Country` của dòng đầu tiên |
| `hour` | giờ của `InvoiceDate` sớm nhất |

In [ ]:
def invoice_summary(lines: pd.DataFrame, col: str = "Customer ID") -> pd.DataFrame:
    """Một dòng mỗi hoá đơn: lines, revenue, missing, mixed, country, hour."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bức tranh tổng
inv = invoice_summary(valid)
miss_lines = valid["Customer ID"].isna()
print(f"dòng thiếu mã: {miss_lines.mean():.1%}; doanh thu trên dòng thiếu mã: "
      f"{(valid.Quantity * valid.Price)[miss_lines].sum() / (valid.Quantity * valid.Price).sum():.1%}")
print(f"hoá đơn: {len(inv):,}; thiếu mã: {inv.missing.sum():,} ({inv.missing.mean():.1%}); lẫn lộn: {inv.mixed.sum()}")
inv.groupby("missing")[["lines", "revenue"]].agg(["median", "mean"]).round(1)

In [ ]:
# explore: thiếu theo quốc gia và theo giờ trong ngày
print(missing_profile(valid, "Customer ID", "Country").head(8).round(3))
by_hour = missing_profile(valid.assign(hour=valid.InvoiceDate.dt.hour), "Customer ID", "hour").sort_index()
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(by_hour.index, by_hour["missing_rate"], color="#4c78a8")
ax.set_xlabel("giờ trong ngày")
ax.set_ylabel("tỉ lệ dòng thiếu mã")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("TODO: headline sau khi đọc chart", loc="left")
plt.show()

## 3. Điều tra tiếp, rồi viết memo

Gợi ý hướng điều tra (thêm cell `explore`, chọn chart cho memo):

- Hoá đơn thiếu mã khác hoá đơn có mã thế nào: số dòng, giá trị, `Quantity` mỗi dòng, `Price` mỗi đơn vị, quốc gia,
  giờ, theo tháng? Một kiểm định (vd. Mann-Whitney cho giá trị hoá đơn, chi-square cho quốc gia) giúp nói "khác" có ý
  nghĩa thống kê, nhưng với cỡ mẫu này, câu hỏi chính là **khác bao nhiêu**.
- Nếu bỏ hết dòng thiếu mã: doanh thu, số hoá đơn, giá trị đơn trung bình (AOV) thay đổi thế nào? Chỉ số nào **cần**
  mã khách, chỉ số nào không?
- Điền mã (một mã "GUEST", hoặc mã khách gần nhất) làm gì với số khách, AOV mỗi khách, tỉ lệ quay lại?

Viết `starter/MEMO.md` theo khung có sẵn (250-700 từ, dòng đầu là headline có con số).